In [1]:
"""
01_data_generation.py
----------------------
Generates a synthetic customer + transaction dataset for the
CLV (Customer Lifetime Value) prediction project.

Outputs:
    data/raw/customers.csv     - one row per customer
    data/raw/transactions.csv  - one row per purchase event

The channel "behavior profiles" below intentionally build in real
patterns (e.g. Referral customers buy more often and spend more)
so that later modeling and statistical tests have genuine signal
to discover, not just noise.
"""

import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

# ----------------------------------------------------------------
# 1. PARAMETERS - tweak these to change the dataset's size/shape
# ----------------------------------------------------------------
NUM_CUSTOMERS = 4000
SEED = 42

# Customers are "acquired" at random dates within this window
ACQUISITION_START = datetime(2022, 1, 1)
ACQUISITION_END = datetime(2023, 6, 30)

# No transactions are simulated past this date (defines how much
# history each customer can have - this is what makes the later
# 90-day calibration / 180-day holdout split possible)
OBSERVATION_END = datetime(2024, 6, 30)

CHANNELS = ["Paid Search", "Email", "Referral", "Social", "Organic"]
CHANNEL_WEIGHTS = [0.35, 0.20, 0.10, 0.20, 0.15]  # must sum to 1.0

# Behavior profile per channel - this is the "story" the model
# will later have to discover from the raw transactions.
CHANNEL_PROFILES = {
    "Paid Search": dict(avg_gap=45, gap_std=15, avg_value=1200, value_std=300, churn_prob=0.08),
    "Email":       dict(avg_gap=35, gap_std=12, avg_value=1100, value_std=250, churn_prob=0.06),
    "Referral":    dict(avg_gap=20, gap_std=8,  avg_value=1800, value_std=400, churn_prob=0.03),
    "Social":      dict(avg_gap=50, gap_std=20, avg_value=900,  value_std=200, churn_prob=0.10),
    "Organic":     dict(avg_gap=40, gap_std=15, avg_value=1300, value_std=350, churn_prob=0.05),
}

AGE_BANDS = ["18-24", "25-34", "35-44", "45-54", "55+"]
REGIONS = ["North", "South", "East", "West"]

rng = np.random.default_rng(SEED)


# ----------------------------------------------------------------
# 2. CUSTOMER MASTER TABLE - one row per customer
# ----------------------------------------------------------------
def random_dates(start, end, n):
    delta_days = (end - start).days
    offsets = rng.integers(0, delta_days + 1, size=n)
    return [start + timedelta(days=int(d)) for d in offsets]


customer_ids = [f"CUST_{i:05d}" for i in range(1, NUM_CUSTOMERS + 1)]
acquisition_dates = random_dates(ACQUISITION_START, ACQUISITION_END, NUM_CUSTOMERS)
acquisition_channels = rng.choice(CHANNELS, size=NUM_CUSTOMERS, p=CHANNEL_WEIGHTS)
age_bands = rng.choice(AGE_BANDS, size=NUM_CUSTOMERS)
regions = rng.choice(REGIONS, size=NUM_CUSTOMERS)

customers = pd.DataFrame({
    "customer_id": customer_ids,
    "acquisition_date": acquisition_dates,
    "acquisition_channel": acquisition_channels,
    "age_band": age_bands,
    "region": regions,
})


# ----------------------------------------------------------------
# 3. TRANSACTION GENERATION - one row per purchase event
# ----------------------------------------------------------------
transaction_rows = []

for _, row in customers.iterrows():
    profile = CHANNEL_PROFILES[row["acquisition_channel"]]
    current_date = row["acquisition_date"]  # first purchase = the conversion event

    while current_date <= OBSERVATION_END:
        amount = max(100, rng.normal(profile["avg_value"], profile["value_std"]))
        transaction_rows.append({
            "customer_id": row["customer_id"],
            "transaction_date": current_date,
            "amount": round(float(amount), 2),
        })

        # Some customers churn after any given purchase and never return
        if rng.random() < profile["churn_prob"]:
            break

        gap_days = max(1, rng.normal(profile["avg_gap"], profile["gap_std"]))
        current_date = current_date + timedelta(days=int(gap_days))

transactions = pd.DataFrame(transaction_rows)


# ----------------------------------------------------------------
# 4. SAVE OUTPUTS
# ----------------------------------------------------------------
os.makedirs("data/raw", exist_ok=True)
customers.to_csv("data/raw/customers.csv", index=False)
transactions.to_csv("data/raw/transactions.csv", index=False)


# ----------------------------------------------------------------
# 5. SANITY CHECKS - confirm the built-in patterns actually show up
# ----------------------------------------------------------------
print(f"Customers generated: {len(customers)}")
print(f"Transactions generated: {len(transactions)}")
print(f"Avg transactions per customer: {len(transactions) / len(customers):.2f}\n")

merged = transactions.merge(customers[["customer_id", "acquisition_channel"]], on="customer_id")
summary = merged.groupby("acquisition_channel").agg(
    avg_order_value=("amount", "mean"),
    purchase_count=("amount", "count"),
    customer_count=("customer_id", "nunique"),
)
summary["avg_purchases_per_customer"] = summary["purchase_count"] / summary["customer_count"]
print("Per-channel summary (confirms Referral should show highest value & frequency):")
print(summary.round(2))

Customers generated: 4000
Transactions generated: 42241
Avg transactions per customer: 10.56

Per-channel summary (confirms Referral should show highest value & frequency):
                     avg_order_value  purchase_count  customer_count  \
acquisition_channel                                                    
Email                        1097.64            8889             783   
Organic                      1304.18            6349             569   
Paid Search                  1196.75           12892            1468   
Referral                     1802.57            8015             398   
Social                        900.48            6096             782   

                     avg_purchases_per_customer  
acquisition_channel                              
Email                                     11.35  
Organic                                   11.16  
Paid Search                                8.78  
Referral                                  20.14  
Social                